# Tutorial 7 - Denoising Autoencoder with Keras (MNIST)

This notebook follows the tutorial example using the **MNIST handwritten digit dataset**:
1. Load and preprocess MNIST
2. Add Gaussian noise
3. Visualize clean and noisy images
4. Build a convolutional denoising autoencoder
5. Train the model
6. Plot training and validation loss
7. Show noisy, denoised, and clean outputs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

## Step 1: Load and preprocess MNIST

In [ ]:
# Load MNIST
(x_train, _), (x_test, _) = mnist.load_data()

# Normalize to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape to (28, 28, 1)
x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))

print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)

## Step 2: Add Gaussian noise

In [ ]:
noise_factor_low = 0.1
noise_factor_high = 0.5

# Low-noise images
x_train_noisy_low = x_train + noise_factor_low * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy_low = x_test + noise_factor_low * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)

# High-noise images
x_train_noisy_high = x_train + noise_factor_high * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy_high = x_test + noise_factor_high * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)

# Clip values to stay in [0, 1]
x_train_noisy_low = np.clip(x_train_noisy_low, 0.0, 1.0)
x_test_noisy_low = np.clip(x_test_noisy_low, 0.0, 1.0)
x_train_noisy_high = np.clip(x_train_noisy_high, 0.0, 1.0)
x_test_noisy_high = np.clip(x_test_noisy_high, 0.0, 1.0)

## Step 3: Visualize original and noisy images

In [ ]:
n = 3
plt.figure(figsize=(9, 9))

for i in range(n):
    # Original
    ax = plt.subplot(n, 3, i * 3 + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Low noise
    ax = plt.subplot(n, 3, i * 3 + 2)
    plt.imshow(x_train_noisy_low[i].reshape(28, 28), cmap="gray")
    plt.title("Noise 0.1")
    plt.axis("off")

    # High noise
    ax = plt.subplot(n, 3, i * 3 + 3)
    plt.imshow(x_train_noisy_high[i].reshape(28, 28), cmap="gray")
    plt.title("Noise 0.5")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 4: Build the convolutional denoising autoencoder

In [ ]:
def build_encoder(input_shape=(28, 28, 1)):
    input_img = Input(shape=input_shape, name="input")
    x = Conv2D(32, (3, 3), activation="relu", padding="same")(input_img)
    x = MaxPooling2D((2, 2), padding="same")(x)
    x = Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    encoded = MaxPooling2D((2, 2), padding="same", name="encoded")(x)
    return input_img, encoded

def build_decoder(encoded_input):
    x = Conv2D(64, (3, 3), activation="relu", padding="same")(encoded_input)
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = UpSampling2D((2, 2))(x)
    decoded = Conv2D(1, (3, 3), activation="sigmoid", padding="same", name="decoded")(x)
    return decoded

def build_autoencoder():
    input_img, encoded_output = build_encoder()
    decoded_output = build_decoder(encoded_output)
    autoencoder = Model(inputs=input_img, outputs=decoded_output, name="autoencoder")
    autoencoder.compile(optimizer=Adam(), loss="binary_crossentropy")
    return autoencoder

autoencoder = build_autoencoder()
autoencoder.summary()

## Step 5: Train the autoencoder

In [ ]:
history = autoencoder.fit(
    x_train_noisy_high,
    x_train,
    epochs=10,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy_high, x_test)
)

## Step 6: Plot training and validation loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## Step 7: Evaluate the autoencoder

In [ ]:
denoised_images = autoencoder.predict(x_test_noisy_high)
print("Denoised output shape:", denoised_images.shape)

## Step 8: Show noisy, denoised, and clean images

In [ ]:
def plot_images(noisy_images, denoised_images, clean_images, n=10):
    plt.figure(figsize=(20, 6))
    for i in range(n):
        # Noisy image
        plt.subplot(3, n, i + 1)
        plt.imshow(noisy_images[i].reshape(28, 28), cmap="gray")
        plt.title("Noisy")
        plt.axis("off")

        # Denoised image
        plt.subplot(3, n, i + 1 + n)
        plt.imshow(denoised_images[i].reshape(28, 28), cmap="gray")
        plt.title("Denoised")
        plt.axis("off")

        # Clean image
        plt.subplot(3, n, i + 1 + 2 * n)
        plt.imshow(clean_images[i].reshape(28, 28), cmap="gray")
        plt.title("Clean")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

plot_images(x_test_noisy_high, denoised_images, x_test, n=10)

## Short conclusion

- The model learns to remove Gaussian noise from handwritten digit images.
- The encoder compresses image features.
- The decoder reconstructs a cleaner version of the image.
- Binary crossentropy is used because the image pixels are scaled to the range [0, 1].